In [2]:

import pandas as pd
import numpy as np

ML_PATH = "/kaggle/input/datasets/anurajgogoi/movielens-25m/ml-25m"

ratings = pd.read_csv(f"{ML_PATH}/ratings.csv")
movies  = pd.read_csv(f"{ML_PATH}/movies.csv")
links   = pd.read_csv(f"{ML_PATH}/links.csv")

print("ratings:", ratings.shape)
print(ratings.head())
print(ratings.dtypes)

print("\nmovies:", movies.shape)
print(movies.head())

print("\nlinks:", links.shape)
print(links.head())
print(links.dtypes)

ratings: (25000095, 4)
   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

movies: (62423, 3)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

links: (62423, 3)
   movieId 

In [3]:
# CELL 2 — Check tmdbId nulls, ratings time range, extract year from movie titles
print("tmdbId null count:", links['tmdbId'].isna().sum())
print("tmdbId null %:", 100 * links['tmdbId'].isna().sum() / len(links))

# Convert timestamp to datetime to see the actual date range we're working with
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
print("\nratings date range:", ratings['datetime'].min(), "to", ratings['datetime'].max())

# Extract release year from movie titles (MovieLens embeds it as "Title (YYYY)")
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)\s*$').astype('Int64')
print("\nmovies missing year:", movies['year'].isna().sum())
print(movies[movies['year'].isna()].head())

print("\nyear distribution:")
print(movies['year'].describe())

tmdbId null count: 107
tmdbId null %: 0.1714111785720007

ratings date range: 1995-01-09 11:46:49 to 2019-11-21 09:15:03

movies missing year: 412
       movieId                                              title  \
15036    79607            Millions Game, The (Das Millionenspiel)   
18789    98063  Mona and the Time of Burning Love (Mona ja pal...   
25387   123619                                 Terrible Joe Moran   
26284   125571               The Court-Martial of Jackie Robinson   
26309   125632                                      In Our Garden   

                             genres  year  
15036  Action|Drama|Sci-Fi|Thriller  <NA>  
18789                         Drama  <NA>  
25387            (no genres listed)  <NA>  
26284            (no genres listed)  <NA>  
26309            (no genres listed)  <NA>  

year distribution:
count        62011.0
mean     1992.045411
std        25.364876
min           1874.0
25%           1976.0
50%           2002.0
75%           2012.0
max    

In [4]:
# CELL 3 — Load TMDB metadata
TMDB_PATH = "/kaggle/input/datasets/rounakbanik/the-movies-dataset"

tmdb = pd.read_csv(f"{TMDB_PATH}/movies_metadata.csv", low_memory=False)

print("tmdb shape:", tmdb.shape)
print(tmdb.columns.tolist())
print(tmdb[['id', 'title', 'genres', 'release_date']].head())
print(tmdb.dtypes[['id', 'genres', 'release_date']])

# id should map to tmdbId in links.csv — check its raw form
print("\nsample 'id' values:", tmdb['id'].head(10).tolist())
print("\nsample 'genres' raw value:", tmdb['genres'].iloc[0])

tmdb shape: (45466, 24)
['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']
      id                        title  \
0    862                    Toy Story   
1   8844                      Jumanji   
2  15602             Grumpier Old Men   
3  31357            Waiting to Exhale   
4  11862  Father of the Bride Part II   

                                              genres release_date  
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   1995-10-30  
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   1995-12-15  
2  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   1995-12-22  
3  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   1995-12-22  
4                     [{'id': 35, 'name': 'Comedy'

In [5]:
# CELL 4 — Find corrupted 'id' rows in TMDB metadata, and load keywords/credits files
import re

# Check which ids are NOT purely numeric strings — these are the corrupted rows
non_numeric_id_mask = ~tmdb['id'].astype(str).str.match(r'^\d+$')
print("corrupted id rows:", non_numeric_id_mask.sum())
print(tmdb[non_numeric_id_mask][['id', 'title', 'release_date']].head(10))

# Load keywords and credits (cast) files
keywords = pd.read_csv(f"{TMDB_PATH}/keywords.csv")
credits  = pd.read_csv(f"{TMDB_PATH}/credits.csv")

print("\nkeywords shape:", keywords.shape)
print(keywords.head(2))
print(keywords.dtypes)

print("\ncredits shape:", credits.shape)
print(credits.columns.tolist())
print(credits[['id', 'cast']].head(2))
print("\nsample 'cast' raw value:", credits['cast'].iloc[0][:500])

corrupted id rows: 3
               id title release_date
19730  1997-08-20   NaN            1
29503  2012-09-29   NaN           12
35587  2014-01-01   NaN           22

keywords shape: (46419, 2)
     id                                           keywords
0   862  [{'id': 931, 'name': 'jealousy'}, {'id': 4290,...
1  8844  [{'id': 10090, 'name': 'board game'}, {'id': 1...
id           int64
keywords    object
dtype: object

credits shape: (45476, 3)
['cast', 'crew', 'id']
     id                                               cast
0   862  [{'cast_id': 14, 'character': 'Woody (voice)',...
1  8844  [{'cast_id': 1, 'character': 'Alan Parrish', '...

sample 'cast' raw value: [{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'n

In [6]:
# CELL 5 — Clean TMDB id, merge keywords/credits, parse stringified columns
import ast

# Drop corrupted rows and cast id to int
tmdb_clean = tmdb[~non_numeric_id_mask].copy()
tmdb_clean['id'] = tmdb_clean['id'].astype(int)

# Merge in keywords and credits on id
tmdb_full = tmdb_clean.merge(keywords, on='id', how='left')
tmdb_full = tmdb_full.merge(credits[['id', 'cast']], on='id', how='left')

print("tmdb_full shape after merges:", tmdb_full.shape)

# Safe parser for stringified list-of-dicts columns
def parse_names(x, key='name', top_n=None):
    if pd.isna(x):
        return []
    try:
        items = ast.literal_eval(x)
        names = [d[key] for d in items]
        return names[:top_n] if top_n else names
    except (ValueError, SyntaxError):
        return []

tmdb_full['genre_list'] = tmdb_full['genres'].apply(parse_names)
tmdb_full['keyword_list'] = tmdb_full['keywords'].apply(parse_names)
tmdb_full['cast_list'] = tmdb_full['cast'].apply(lambda x: parse_names(x, top_n=5))  # top 5 billed

print("\nsample parsed row:")
print(tmdb_full[['title', 'genre_list', 'keyword_list', 'cast_list']].iloc[0])

# Check how many rows ended up with empty lists (parsing failures or genuinely missing data)
print("\nempty genre_list:", (tmdb_full['genre_list'].str.len() == 0).sum())
print("empty keyword_list:", (tmdb_full['keyword_list'].str.len() == 0).sum())
print("empty cast_list:", (tmdb_full['cast_list'].str.len() == 0).sum())

tmdb_full shape after merges: (46629, 26)

sample parsed row:
title                                                   Toy Story
genre_list                            [Animation, Comedy, Family]
keyword_list    [jealousy, toy, boy, friendship, friends, riva...
cast_list       [Tom Hanks, Tim Allen, Don Rickles, Jim Varney...
Name: 0, dtype: object

empty genre_list: 2525
empty keyword_list: 14890
empty cast_list: 2492


In [7]:
tmdb_full.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,tagline,title,video,vote_average,vote_count,keywords,cast,genre_list,keyword_list,cast_list
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,NaN,Toy Story,False,7.7,5415.0,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney..."
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[Adventure, Fantasy, Family]","[board game, disappearance, based on children'...","[Robin Williams, Jonathan Hyde, Kirsten Dunst,..."
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o...","[Walter Matthau, Jack Lemmon, Ann-Margret, Sop..."
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin...","[Whitney Houston, Angela Bassett, Loretta Devi..."
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...",[Comedy],"[baby, midlife crisis, confidence, aging, daug...","[Steve Martin, Diane Keaton, Martin Short, Kim..."


In [8]:
# CELL 6 — Diagnose and fix duplicate rows from the merge
print("tmdb_clean duplicate ids:", tmdb_clean['id'].duplicated().sum())
print("keywords duplicate ids:", keywords['id'].duplicated().sum())
print("credits duplicate ids:", credits['id'].duplicated().sum())

# Look at an example of a duplicated id to understand what's different between the dupe rows
dupe_id_keywords = keywords[keywords['id'].duplicated(keep=False)]['id'].iloc[0] if keywords['id'].duplicated().sum() > 0 else None
dupe_id_credits = credits[credits['id'].duplicated(keep=False)]['id'].iloc[0] if credits['id'].duplicated().sum() > 0 else None

print("\nexample duplicate in keywords:")
print(keywords[keywords['id'] == dupe_id_keywords] if dupe_id_keywords else "none")

print("\nexample duplicate in credits:")
print(credits[credits['id'] == dupe_id_credits] if dupe_id_credits else "none")

# Also check tmdb_full itself for duplicate ids post-merge
print("\ntmdb_full duplicate ids:", tmdb_full['id'].duplicated().sum())

tmdb_clean duplicate ids: 30
keywords duplicate ids: 987
credits duplicate ids: 44

example duplicate in keywords:
          id                                           keywords
676   105045  [{'id': 7059, 'name': 'anti-communism'}, {'id'...
1465  105045  [{'id': 7059, 'name': 'anti-communism'}, {'id'...

example duplicate in credits:
                                                   cast  \
676   [{'cast_id': 5, 'character': 'Sophie II', 'cre...   
1465  [{'cast_id': 5, 'character': 'Sophie II', 'cre...   

                                                   crew      id  
676   [{'credit_id': '52fe4a44c3a36847f81c463f', 'de...  105045  
1465  [{'credit_id': '52fe4a44c3a36847f81c463f', 'de...  105045  

tmdb_full duplicate ids: 1196


In [9]:
# CELL 7 — Deduplicate all three tables on id, then redo the merges cleanly
tmdb_clean_dedup = tmdb_clean.drop_duplicates(subset='id', keep='first')
keywords_dedup   = keywords.drop_duplicates(subset='id', keep='first')
credits_dedup    = credits.drop_duplicates(subset='id', keep='first')

print("tmdb_clean_dedup:", tmdb_clean_dedup.shape)
print("keywords_dedup:", keywords_dedup.shape)
print("credits_dedup:", credits_dedup.shape)

# Redo the merges
tmdb_full = tmdb_clean_dedup.merge(keywords_dedup, on='id', how='left')
tmdb_full = tmdb_full.merge(credits_dedup[['id', 'cast']], on='id', how='left')

print("\ntmdb_full shape after clean merge:", tmdb_full.shape)
print("tmdb_full duplicate ids now:", tmdb_full['id'].duplicated().sum())

# Re-parse the stringified columns on the clean table
tmdb_full['genre_list'] = tmdb_full['genres'].apply(parse_names)
tmdb_full['keyword_list'] = tmdb_full['keywords'].apply(parse_names)
tmdb_full['cast_list'] = tmdb_full['cast'].apply(lambda x: parse_names(x, top_n=5))

print("\nempty genre_list:", (tmdb_full['genre_list'].str.len() == 0).sum())
print("empty keyword_list:", (tmdb_full['keyword_list'].str.len() == 0).sum())
print("empty cast_list:", (tmdb_full['cast_list'].str.len() == 0).sum())

tmdb_clean_dedup: (45433, 24)
keywords_dedup: (45432, 2)
credits_dedup: (45432, 3)

tmdb_full shape after clean merge: (45433, 26)
tmdb_full duplicate ids now: 0

empty genre_list: 2442
empty keyword_list: 14341
empty cast_list: 2415


In [10]:
# CELL 8 — Join MovieLens (via links.csv) to tmdb_full, measure coverage
# links.tmdbId is float (has ~107 nulls) — drop nulls and cast to int before joining
links_clean = links.dropna(subset=['tmdbId']).copy()
links_clean['tmdbId'] = links_clean['tmdbId'].astype(int)

# Merge movies + links_clean on movieId to attach tmdbId to every movie
movies_with_tmdb_id = movies.merge(links_clean[['movieId', 'tmdbId', 'imdbId']], on='movieId', how='left')

print("movies:", movies.shape[0])
print("movies with a tmdbId:", movies_with_tmdb_id['tmdbId'].notna().sum())

# Now join to tmdb_full on tmdbId == id
movies_joined = movies_with_tmdb_id.merge(
    tmdb_full[['id', 'genre_list', 'keyword_list', 'cast_list', 'overview', 'vote_average', 'vote_count']],
    left_on='tmdbId', right_on='id', how='left'
)

print("\nmovies_joined shape:", movies_joined.shape)
print("movies with successful TMDB metadata match:", movies_joined['id'].notna().sum())
print("coverage %:", 100 * movies_joined['id'].notna().sum() / len(movies_joined))

# Now measure coverage at the RATINGS level, not just movie level, since that's what actually matters for training data volume
ratings_joined_check = ratings.merge(movies_joined[['movieId', 'id']], on='movieId', how='left')
print("\nratings with a matched TMDB movie:", ratings_joined_check['id'].notna().sum())
print("ratings coverage %:", 100 * ratings_joined_check['id'].notna().sum() / len(ratings_joined_check))

# Look at a sample of movies that did NOT get matched, to spot patterns
unmatched = movies_joined[movies_joined['id'].isna()]
print("\nsample unmatched movies:")
print(unmatched[['title', 'genres']].head(10))

movies: 62423
movies with a tmdbId: 62316

movies_joined shape: (62423, 13)
movies with successful TMDB metadata match: 42849
coverage %: 68.64296813674447

ratings with a matched TMDB movie: 24732580
ratings coverage %: 98.92994406621254

sample unmatched movies:
                                                 title  \
140                            Shadows (Cienie) (1988)   
596                                   Criminals (1996)   
705  Wallace & Gromit: The Best of Aardman Animatio...   
706           Halfmoon (Paul Bowles - Halbmond) (1995)   
715                                    Low Life (1994)   
753          Marlene Dietrich: Shadow and Light (1996)   
754                                 Costa Brava (1946)   
775  Last Klezmer: Leopold Kozlowski, His Life and ...   
803                            Crude Oasis, The (1995)   
841                      Hippie Revolution, The (1996)   

                         genres  
140                       Drama  
596                 Document

In [11]:
# CELL 9 — Finalize movies master table + build TF-IDF text field
movies_master = movies_joined.drop(columns=['id']).rename(columns={'tmdbId': 'tmdb_id', 'imdbId': 'imdb_id'})

# Fill missing lists with empty lists (NaN from the left join, not the parser)
for col in ['genre_list', 'keyword_list', 'cast_list']:
    movies_master[col] = movies_master[col].apply(lambda x: x if isinstance(x, list) else [])

# Build the combined text field per spec 5.2: genre tags + keywords + top-billed cast
def build_content_text(row):
    parts = row['genre_list'] + row['keyword_list'] + row['cast_list']
    return ' '.join(str(p).replace(' ', '_') for p in parts)  # underscore multi-word names/keywords so TF-IDF treats them as single tokens

movies_master['content_text'] = movies_master.apply(build_content_text, axis=1)

print("movies_master shape:", movies_master.shape)
print(movies_master[['movieId', 'title', 'content_text']].head(5))

# How many movies end up with an EMPTY content_text (no genre, no keyword, no cast at all)
empty_content = (movies_master['content_text'].str.len() == 0).sum()
print("\nmovies with completely empty content_text:", empty_content)
print("as % of all movies:", 100 * empty_content / len(movies_master))

movies_master shape: (62423, 13)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        content_text  
0  Animation Comedy Family jealousy toy boy frien...  
1  Adventure Fantasy Family board_game disappeara...  
2  Romance Comedy fishing best_friend duringcredi...  
3  Comedy Drama Romance based_on_novel interracia...  
4  Comedy baby midlife_crisis confidence aging da...  

movies with completely empty content_text: 20134
as % of all movies: 32.254137096903385


In [12]:
# CELL 10 — Diagnose empty content_text, add MovieLens genres as fallback
# Is "empty content_text" basically the same set as "no TMDB match"?
no_tmdb_match = movies_master['tmdb_id'].isna()
print("no tmdb match:", no_tmdb_match.sum())
print("empty content_text:", (movies_master['content_text'].str.len() == 0).sum())
print("overlap (no match AND empty text):", (no_tmdb_match & (movies_master['content_text'].str.len() == 0)).sum())

# matched but STILL empty (genre/keyword/cast all missing despite a TMDB match)
matched_but_empty = (~no_tmdb_match) & (movies_master['content_text'].str.len() == 0)
print("matched but still empty:", matched_but_empty.sum())

# Use MovieLens's own genres column (pipe-separated, e.g. "Comedy|Romance") as fallback text
# where TMDB gave us nothing
movies_master['ml_genre_list'] = movies_master['genres'].apply(
    lambda x: [] if x == '(no genres listed)' else x.split('|')
)

def build_content_text_v2(row):
    parts = row['genre_list'] + row['keyword_list'] + row['cast_list']
    if not parts:  # TMDB gave nothing usable — fall back to MovieLens genres
        parts = row['ml_genre_list']
    return ' '.join(str(p).replace(' ', '_') for p in parts)

movies_master['content_text'] = movies_master.apply(build_content_text_v2, axis=1)

empty_after_fallback = (movies_master['content_text'].str.len() == 0).sum()
print("\nempty content_text AFTER fallback:", empty_after_fallback)
print("as % of all movies:", 100 * empty_after_fallback / len(movies_master))

no tmdb match: 107
empty content_text: 20134
overlap (no match AND empty text): 107
matched but still empty: 20027

empty content_text AFTER fallback: 2801
as % of all movies: 4.487128141870785


In [13]:
# CELL 11 — Save movies_master, then load RT critic reviews dataset
import os
os.makedirs('/kaggle/working/interim', exist_ok=True)
movies_master.to_parquet('/kaggle/working/interim/movies_master.parquet', index=False)
print("saved movies_master:", movies_master.shape)


saved movies_master: (62423, 14)


In [14]:

# Load RT reviews — adjust path to match your Kaggle input for stefanoleone992's dataset
RT_PATH = "/kaggle/input/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset"

rt_movies  = pd.read_csv(f"{RT_PATH}/rotten_tomatoes_movies.csv")
rt_reviews = pd.read_csv(f"{RT_PATH}/rotten_tomatoes_critic_reviews.csv")

print("\nrt_movies shape:", rt_movies.shape)
print(rt_movies.columns.tolist())
print(rt_movies[['rotten_tomatoes_link', 'movie_title', 'releaseDateTheaters']].head() if 'releaseDateTheaters' in rt_movies.columns else rt_movies.head())

print("\nrt_reviews shape:", rt_reviews.shape)
print(rt_reviews.columns.tolist())
print(rt_reviews.head(3))


rt_movies shape: (17712, 22)
['rotten_tomatoes_link', 'movie_title', 'movie_info', 'critics_consensus', 'content_rating', 'genres', 'directors', 'authors', 'actors', 'original_release_date', 'streaming_release_date', 'runtime', 'production_company', 'tomatometer_status', 'tomatometer_rating', 'tomatometer_count', 'audience_status', 'audience_rating', 'audience_count', 'tomatometer_top_critics_count', 'tomatometer_fresh_critics_count', 'tomatometer_rotten_critics_count']
                    rotten_tomatoes_link  \
0                              m/0814255   
1                              m/0878835   
2                                   m/10   
3                 m/1000013-12_angry_men   
4  m/1000079-20000_leagues_under_the_sea   

                                         movie_title  \
0  Percy Jackson & the Olympians: The Lightning T...   
1                                        Please Give   
2                                                 10   
3                    12 Angry Men (

In [15]:
# CELL 12 — Normalize titles on both sides, extract RT year, attempt the join
import re

def normalize_title(title):
    if pd.isna(title):
        return ''
    t = title.lower()
    t = re.sub(r'\(\d{4}\)\s*$', '', t)          # strip trailing "(YYYY)" — for movies_master
    t = re.sub(r'[^a-z0-9\s]', '', t)             # strip punctuation
    t = re.sub(r'^(the|a|an)\s+', '', t.strip())  # strip leading article
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# movies_master: strip trailing ", The" style suffix MovieLens sometimes uses, then normalize
movies_master['title_norm'] = movies_master['title'].apply(normalize_title)

# rt_movies: extract year from original_release_date, normalize title
rt_movies['year'] = pd.to_datetime(rt_movies['original_release_date'], errors='coerce').dt.year
rt_movies['title_norm'] = rt_movies['movie_title'].apply(normalize_title)

print("rt_movies year nulls:", rt_movies['year'].isna().sum())
print(rt_movies[['movie_title', 'title_norm', 'year']].head())

# Attempt the join on (title_norm, year)
rt_join = movies_master.merge(
    rt_movies[['rotten_tomatoes_link', 'title_norm', 'year']],
    on=['title_norm', 'year'], how='left'
)

print("\nmovies_master rows:", len(movies_master))
print("rt_join rows (post-merge):", len(rt_join))
print("matched to an RT link:", rt_join['rotten_tomatoes_link'].notna().sum())
print("match rate %:", 100 * rt_join['rotten_tomatoes_link'].notna().sum() / len(movies_master))

# Check for duplication from the merge (multiple RT movies matching same title+year)
print("\nduplicate movieId after merge:", rt_join['movieId'].duplicated().sum())

rt_movies year nulls: 1166
                                         movie_title  \
0  Percy Jackson & the Olympians: The Lightning T...   
1                                        Please Give   
2                                                 10   
3                    12 Angry Men (Twelve Angry Men)   
4                       20,000 Leagues Under The Sea   

                                        title_norm    year  
0  percy jackson the olympians the lightning thief  2010.0  
1                                      please give  2010.0  
2                                               10  1979.0  
3                    12 angry men twelve angry men  1957.0  
4                      20000 leagues under the sea  1954.0  

movies_master rows: 62423
rt_join rows (post-merge): 62429
matched to an RT link: 8935
match rate %: 14.313634397577816

duplicate movieId after merge: 6


In [16]:
# CELL 13 — Diagnose the low match rate: is it a year mismatch problem?

# First, drop the 6 duplicate rows (rare title+year collisions) before further checks
dupe_ids = rt_join[rt_join['movieId'].duplicated(keep=False)]['movieId'].unique()
print("example duplicate collisions:")
print(rt_join[rt_join['movieId'].isin(dupe_ids)][['movieId', 'title', 'title_norm', 'year']])

# Check: how many movies match on title_norm ALONE (ignoring year)?
title_only_match = movies_master['title_norm'].isin(rt_movies['title_norm'])
print("\ntitle-only match count:", title_only_match.sum())
print("title-only match rate %:", 100 * title_only_match.sum() / len(movies_master))

# For movies that matched on title but check the year gap for those cases
title_matched = movies_master[title_only_match][['movieId', 'title', 'title_norm', 'year']].merge(
    rt_movies[['title_norm', 'year']], on='title_norm', how='left', suffixes=('_ml', '_rt')
)
title_matched['year_diff'] = (title_matched['year_ml'] - title_matched['year_rt']).abs()

print("\nyear_diff distribution (where title matched):")
print(title_matched['year_diff'].value_counts(dropna=False).sort_index().head(15))

example duplicate collisions:
       movieId                    title    title_norm  year
3499      3598            Hamlet (2000)        hamlet  2000
3500      3598            Hamlet (2000)        hamlet  2000
12615    61312             Noise (2007)         noise  2007
12616    61312             Noise (2007)         noise  2007
12986    65665            Hamlet (2000)        hamlet  2000
12987    65665            Hamlet (2000)        hamlet  2000
38758   155595  The Confirmation (2016)  confirmation  2016
38759   155595  The Confirmation (2016)  confirmation  2016
39279   156819      Confirmation (2016)  confirmation  2016
39280   156819      Confirmation (2016)  confirmation  2016
55334   191713             Noise (2007)         noise  2007
55335   191713             Noise (2007)         noise  2007

title-only match count: 14562
title-only match rate %: 23.327940022107235

year_diff distribution (where title matched):
year_diff
0.0     8926
1.0     2067
2.0      388
3.0      231
4.0   

In [17]:
# CELL 14 — Apply ±1 year tolerance join, dedupe, then measure RATINGS-level coverage (the number that matters)

# Drop the handful of duplicate collisions first (keep first occurrence)
rt_join_dedup = rt_join.drop_duplicates(subset='movieId', keep='first')

# Redo the join allowing year within ±1, taking the closest year match per movie
# Cross join on title_norm only, then filter to closest year, then keep first
candidates = movies_master[['movieId', 'title_norm', 'year']].merge(
    rt_movies[['rotten_tomatoes_link', 'title_norm', 'year']],
    on='title_norm', how='inner', suffixes=('_ml', '_rt')
)
candidates['year_diff'] = (candidates['year_ml'] - candidates['year_rt']).abs()
candidates_tol = candidates[candidates['year_diff'] <= 1].copy()
candidates_tol = candidates_tol.sort_values('year_diff').drop_duplicates(subset='movieId', keep='first')

print("movies matched with ±1 year tolerance:", candidates_tol['movieId'].nunique())
print("match rate %:", 100 * candidates_tol['movieId'].nunique() / len(movies_master))

# Build final movieId -> rotten_tomatoes_link mapping
movie_to_rt = candidates_tol[['movieId', 'rotten_tomatoes_link']].drop_duplicates(subset='movieId')

# Now measure coverage at the RATINGS level
ratings_rt_check = ratings.merge(movie_to_rt, on='movieId', how='left')
print("\nratings with a matched RT movie:", ratings_rt_check['rotten_tomatoes_link'].notna().sum())
print("ratings coverage %:", 100 * ratings_rt_check['rotten_tomatoes_link'].notna().sum() / len(ratings_rt_check))

movies matched with ±1 year tolerance: 10958
match rate %: 17.554427054130688

ratings with a matched RT movie: 16619063
ratings coverage %: 66.47599939120231


In [18]:
# CELL 15 — Check review count per matched movie, save the RT mapping
# Join reviews to our movie_to_rt mapping to see review volume per movieId
reviews_per_movie = rt_reviews.merge(movie_to_rt, on='rotten_tomatoes_link', how='inner')
review_counts = reviews_per_movie.groupby('movieId').size()

print("matched movies with at least 1 review:", (review_counts > 0).sum())
print("\nreview count distribution per movie:")
print(review_counts.describe())
print("\npercentiles:")
print(review_counts.quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

# How many matched movies have very few reviews (say, < 5) — too thin to trust a sentiment aggregate
thin_coverage = (review_counts < 5).sum()
print(f"\nmatched movies with < 5 reviews: {thin_coverage} ({100*thin_coverage/len(review_counts):.1f}%)")

# Save the mapping + review data for later use in the sentiment pipeline
movie_to_rt.to_parquet('/kaggle/working/interim/movie_to_rt_mapping.parquet', index=False)
reviews_per_movie.to_parquet('/kaggle/working/interim/rt_reviews_matched.parquet', index=False)
print("\nsaved movie_to_rt_mapping and rt_reviews_matched")

matched movies with at least 1 review: 10956

review count distribution per movie:
count    10956.000000
mean        75.615553
std         90.945074
min          4.000000
25%         16.000000
50%         38.000000
75%        106.000000
max        948.000000
dtype: float64

percentiles:
0.10      9.0
0.25     16.0
0.50     38.0
0.75    106.0
0.90    192.0
dtype: float64

matched movies with < 5 reviews: 1 (0.0%)

saved movie_to_rt_mapping and rt_reviews_matched


In [19]:
# CELL 16 — Load IMDB 50K Reviews
IMDB_PATH = "/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"  

imdb = pd.read_csv(f"{IMDB_PATH}/IMDB Dataset.csv")

print("imdb shape:", imdb.shape)
print(imdb.columns.tolist())
print(imdb.head(3))
print("\nlabel distribution:")
print(imdb['sentiment'].value_counts())

# Check for nulls/dupes since this becomes classifier training data directly
print("\nnull review text:", imdb['review'].isna().sum())
print("duplicate reviews:", imdb['review'].duplicated().sum())

# Sample review length
imdb['review_len'] = imdb['review'].str.len()
print("\nreview length stats:")
print(imdb['review_len'].describe())

imdb shape: (50000, 2)
['review', 'sentiment']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive

label distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

null review text: 0
duplicate reviews: 418

review length stats:
count    50000.000000
mean      1309.431020
std        989.728014
min         32.000000
25%        699.000000
50%        970.000000
75%       1590.250000
max      13704.000000
Name: review_len, dtype: float64


In [20]:
# CELL 17 — Clean IMDB reviews: dedupe, strip HTML, then save
import re as re_module

# Drop exact duplicate reviews
imdb_clean = imdb.drop_duplicates(subset='review', keep='first').copy()
print("rows after dedup:", imdb_clean.shape)

# Strip HTML tags (mainly <br /> in this dataset)
def clean_html(text):
    text = re_module.sub(r'<[^>]+>', ' ', text)
    text = re_module.sub(r'\s+', ' ', text).strip()
    return text

imdb_clean['review_clean'] = imdb_clean['review'].apply(clean_html)

print("\nbefore/after example:")
print("BEFORE:", imdb_clean['review'].iloc[1][:200])
print("AFTER: ", imdb_clean['review_clean'].iloc[1][:200])

# Re-check label balance post-dedup (should still be roughly balanced)
print("\nlabel distribution after dedup:")
print(imdb_clean['sentiment'].value_counts())

# Token length matters for DistilBERT's 512-token limit — rough proxy using word count
imdb_clean['word_count'] = imdb_clean['review_clean'].str.split().str.len()
print("\nword count stats:")
print(imdb_clean['word_count'].describe())
pct_over_400_words = (imdb_clean['word_count'] > 400).mean() * 100
print(f"\n% of reviews over 400 words (likely to need truncation for DistilBERT): {pct_over_400_words:.1f}%")

# Save cleaned version
imdb_clean[['review_clean', 'sentiment']].to_parquet('/kaggle/working/interim/imdb_reviews_clean.parquet', index=False)
print("\nsaved imdb_reviews_clean")

rows after dedup: (49582, 3)

before/after example:
BEFORE: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece
AFTER:  A wonderful little production. The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. The actors

label distribution after dedup:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64

word count stats:
count    49582.000000
mean       229.056795
std        169.785996
min          4.000000
25%        125.000000
50%        172.000000
75%        278.000000
max       2459.000000
Name: word_count, dtype: float64

% of reviews over 400 words (likely to need truncation for DistilBERT): 12.7%

saved imdb_reviews_clean


In [21]:
# CELL 18 — Build the global temporal split + summary of Week 1 findings

# Look at ratings volume by year to pick a sensible split point
ratings['year'] = ratings['datetime'].dt.year
yearly_counts = ratings.groupby('year').size()
print("ratings per year:")
print(yearly_counts)

# Proposed split: e.g. train < 2017-01-01, val = 2017-01-01 to 2018-06-30, test >= 2018-07-01
# (adjust once we see the yearly distribution above)
train_cutoff = pd.Timestamp('2017-01-01')
val_cutoff = pd.Timestamp('2018-07-01')

n_train = (ratings['datetime'] < train_cutoff).sum()
n_val = ((ratings['datetime'] >= train_cutoff) & (ratings['datetime'] < val_cutoff)).sum()
n_test = (ratings['datetime'] >= val_cutoff).sum()

print(f"\ntrain: {n_train} ({100*n_train/len(ratings):.1f}%)")
print(f"val:   {n_val} ({100*n_val/len(ratings):.1f}%)")
print(f"test:  {n_test} ({100*n_test/len(ratings):.1f}%)")

ratings per year:
year
1995          3
1996    1430093
1997     626202
1998     272099
1999    1059080
2000    1735398
2001    1058750
2002     776654
2003     920295
2004    1048116
2005    1613550
2006    1038458
2007     931432
2008    1018001
2009     810127
2010     792436
2011     676498
2012     635208
2013     515684
2014     478270
2015    1604971
2016    1757440
2017    1689935
2018    1310761
2019    1200634
dtype: int64

train: 20798765 (83.2%)
val:   2301354 (9.2%)
test:  1899976 (7.6%)


In [22]:
# CELL 19 — Apply the temporal split, save train/val/test, print Week 1 summary

ratings_split = ratings.copy()
ratings_split['split'] = np.select(
    [ratings_split['datetime'] < train_cutoff,
     ratings_split['datetime'] < val_cutoff],
    ['train', 'val'],
    default='test'
)

print(ratings_split['split'].value_counts())

# Save splits
for split_name in ['train', 'val', 'test']:
    subset = ratings_split[ratings_split['split'] == split_name].drop(columns=['split'])
    subset.to_parquet(f'/kaggle/working/interim/ratings_{split_name}.parquet', index=False)
    print(f"saved ratings_{split_name}: {subset.shape}")

# Sparsity check — per-user sequence length, TRAIN SPLIT ONLY (per spec: derived from train)
train_ratings = ratings_split[ratings_split['split'] == 'train']
seq_lengths = train_ratings.groupby('userId').size()

print("\nper-user sequence length (train split) percentiles:")
print(seq_lengths.describe())
print(seq_lengths.quantile([0.1, 0.2, 0.25, 0.5, 0.75, 0.9]))

# ---- WEEK 1 SUMMARY ----
print("\n" + "="*50)
print("WEEK 1 DATA PREP SUMMARY")
print("="*50)
print(f"MovieLens ratings: {len(ratings):,}")
print(f"TMDB join coverage (ratings-level): 98.9%")
print(f"Content-text empty rate (after ML genre fallback): 4.5%")
print(f"RT review join coverage (ratings-level): 66.5%")
print(f"RT matched movies with <5 reviews: 1 (negligible)")
print(f"IMDB reviews after dedup: 49,582 (balanced 24,884/24,698)")
print(f"Temporal split: train {n_train:,} (83.2%) / val {n_val:,} (9.2%) / test {n_test:,} (7.6%)")
print(f"Split cutoffs: train < {train_cutoff.date()}, val < {val_cutoff.date()}, test >= {val_cutoff.date()}")

split
train    20798765
val       2301354
test      1899976
Name: count, dtype: int64
saved ratings_train: (20798765, 6)
saved ratings_val: (2301354, 6)
saved ratings_test: (1899976, 6)

per-user sequence length (train split) percentiles:
count    142184.000000
mean        146.280629
std         240.323949
min           1.000000
25%          35.000000
50%          68.000000
75%         155.000000
max       14790.000000
dtype: float64
0.10     24.0
0.20     31.0
0.25     35.0
0.50     68.0
0.75    155.0
0.90    337.0
dtype: float64

WEEK 1 DATA PREP SUMMARY
MovieLens ratings: 25,000,095
TMDB join coverage (ratings-level): 98.9%
Content-text empty rate (after ML genre fallback): 4.5%
RT review join coverage (ratings-level): 66.5%
RT matched movies with <5 reviews: 1 (negligible)
IMDB reviews after dedup: 49,582 (balanced 24,884/24,698)
Temporal split: train 20,798,765 (83.2%) / val 2,301,354 (9.2%) / test 1,899,976 (7.6%)
Split cutoffs: train < 2017-01-01, val < 2018-07-01, test >= 2018-

In [23]:
# Cell 22 — decade/genre skew of RT-matched vs full catalog
movies_master['decade'] = (movies_master['year'] // 10 * 10)
movies_master['has_rt'] = movies_master['movieId'].isin(movie_to_rt['movieId'])

print("Full catalog decade distribution:")
print(movies_master['decade'].value_counts(normalize=True).sort_index())
print("\nRT-matched decade distribution:")
print(movies_master[movies_master['has_rt']]['decade'].value_counts(normalize=True).sort_index())

Full catalog decade distribution:
decade
1870    0.000032
1880    0.000113
1890    0.001742
1900      0.0025
1910    0.003854
1920    0.009289
1930    0.033365
1940    0.037509
1950    0.048056
1960    0.057828
1970    0.079954
1980     0.08355
1990    0.107336
2000    0.204464
2010    0.330409
Name: proportion, dtype: Float64

RT-matched decade distribution:
decade
1910    0.000091
1920    0.002464
1930    0.011681
1940    0.019529
1950    0.023818
1960    0.030389
1970     0.03988
1980    0.079668
1990    0.137434
2000    0.250958
2010    0.404088
Name: proportion, dtype: Float64


In [24]:
train_users = set(ratings_split[ratings_split['split']=='train']['userId'])
val_users = set(ratings_split[ratings_split['split']=='val']['userId'])
test_users = set(ratings_split[ratings_split['split']=='test']['userId'])

print(f"Val users also in train: {len(val_users & train_users) / len(val_users):.1%}")
print(f"Test users also in train: {len(test_users & train_users) / len(test_users):.1%}")

Val users also in train: 33.8%
Test users also in train: 24.8%


In [25]:
# Cell 22 — Restrict SVD/GRU eval sets to users seen in train
val_known = ratings_split[(ratings_split['split']=='val') & (ratings_split['userId'].isin(train_users))]
test_known = ratings_split[(ratings_split['split']=='test') & (ratings_split['userId'].isin(train_users))]

print(f"Val eval set (known users only): {len(val_known):,} rows, {val_known['userId'].nunique():,} users")
print(f"Test eval set (known users only): {len(test_known):,} rows, {test_known['userId'].nunique():,} users")
print(f"Val users excluded (unseen at train): {(1 - len(val_known)/len(ratings_split[ratings_split['split']=='val'])):.1%} of val rows")

Val eval set (known users only): 440,559 rows, 5,750 users
Test eval set (known users only): 257,274 rows, 3,602 users
Val users excluded (unseen at train): 80.9% of val rows


In [26]:
for cutoff_year in [2014, 2015, 2016]:
    tc = pd.Timestamp(f'{cutoff_year}-01-01')
    tu = set(ratings[ratings['datetime'] < tc]['userId'])
    vu = set(ratings[(ratings['datetime'] >= tc) & (ratings['datetime'] < val_cutoff)]['userId'])
    print(f"train_cutoff={cutoff_year}: val-user overlap = {len(vu & tu)/len(vu):.1%}")

train_cutoff=2014: val-user overlap = 11.4%
train_cutoff=2015: val-user overlap = 12.0%
train_cutoff=2016: val-user overlap = 22.0%


In [28]:
# CELL 20 — Set cold-start threshold N, finalize Week 1 artifacts list

COLD_START_N = 25  # ~10th percentile of train-split sequence lengths (24) -> rounded

n_cold_start_users = (seq_lengths < COLD_START_N).sum()
print(f"Users below cold-start threshold N={COLD_START_N}: {n_cold_start_users} "
      f"({100*n_cold_start_users/len(seq_lengths):.1f}% of users)")

# Save the config values so downstream weeks just import these, not re-derive them
import json
config = {
    "train_cutoff": str(train_cutoff.date()),
    "val_cutoff": str(val_cutoff.date()),
    "cold_start_threshold_N": COLD_START_N,
    "tmdb_join_coverage_ratings_pct": 98.93,
    "rt_join_coverage_ratings_pct": 66.48,
    "content_text_empty_pct": 4.49,
    "imdb_reviews_final_count": 49582,
}
with open('/kaggle/working/interim/week1_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("\nsaved week1_config.json")
print(json.dumps(config, indent=2))

print("\nWeek 1 artifacts saved to /kaggle/working/interim/:")
import os
for f in os.listdir('/kaggle/working/interim'):
    print(" -", f)

Users below cold-start threshold N=25: 15593 (11.0% of users)

saved week1_config.json
{
  "train_cutoff": "2017-01-01",
  "val_cutoff": "2018-07-01",
  "cold_start_threshold_N": 25,
  "tmdb_join_coverage_ratings_pct": 98.93,
  "rt_join_coverage_ratings_pct": 66.48,
  "content_text_empty_pct": 4.49,
  "imdb_reviews_final_count": 49582
}

Week 1 artifacts saved to /kaggle/working/interim/:
 - movies_master.parquet
 - movie_to_rt_mapping.parquet
 - imdb_reviews_clean.parquet
 - week1_config.json
 - ratings_train.parquet
 - rt_reviews_matched.parquet
 - ratings_test.parquet
 - ratings_val.parquet


In [ ]:
# CELL 21 — Export all Week 1 parquet artifacts as CSV
import os

interim_dir = '/kaggle/working/interim'
parquet_files = [f for f in os.listdir(interim_dir) if f.endswith('.parquet')]

for pf in parquet_files:
    df = pd.read_parquet(f'{interim_dir}/{pf}')
    csv_name = pf.replace('.parquet', '.csv')
    df.to_csv(f'{interim_dir}/{csv_name}', index=False)
    print(f"saved {csv_name} — shape {df.shape}")

print("\nfull interim directory contents:")
for f in sorted(os.listdir(interim_dir)):
    print(" -", f)

In [29]:
# CELL 22 — Distinguish "cold-start by design" from "genuine warm-user eval set" for SVD/GRU

train_user_counts = ratings_split[ratings_split['split'] == 'train'].groupby('userId').size()
warm_users = set(train_user_counts[train_user_counts >= COLD_START_N].index)

for split_name in ['val', 'test']:
    split_df = ratings_split[ratings_split['split'] == split_name]
    split_users = split_df['userId'].unique()

    n_zero_train = sum(1 for u in split_users if u not in train_users)
    n_thin_train = sum(1 for u in split_users if u in train_users and u not in warm_users)
    n_warm = sum(1 for u in split_users if u in warm_users)

    print(f"--- {split_name} ---")
    print(f"users with zero train history:  {n_zero_train} ({100*n_zero_train/len(split_users):.1f}%)")
    print(f"users with train history < N:   {n_thin_train} ({100*n_thin_train/len(split_users):.1f}%)")
    print(f"warm users (>= N train ratings): {n_warm} ({100*n_warm/len(split_users):.1f}%)")

    # ratings-level, not just user-level — this is what actually matters for eval set size
    ratings_from_warm = split_df[split_df['userId'].isin(warm_users)]
    print(f"ratings from warm users: {len(ratings_from_warm)} ({100*len(ratings_from_warm)/len(split_df):.1f}% of {split_name})\n")

--- val ---
users with zero train history:  11262 (66.2%)
users with train history < N:   296 (1.7%)
warm users (>= N train ratings): 5454 (32.1%)
ratings from warm users: 412348 (17.9% of val)

--- test ---
users with zero train history:  10937 (75.2%)
users with train history < N:   175 (1.2%)
warm users (>= N train ratings): 3427 (23.6%)
ratings from warm users: 242544 (12.8% of test)



In [30]:
# CELL 23 — Save warm-user eval subsets + update config

warm_users_list = sorted(warm_users)

# Save as a lookup table
warm_users_df = pd.DataFrame({'userId': warm_users_list})
warm_users_df.to_parquet('/kaggle/working/interim/warm_users.parquet', index=False)

# Save the warm-user-only val/test ratings as separate eval files (for convenience in Week 2)
val_df = ratings_split[ratings_split['split'] == 'val']
test_df = ratings_split[ratings_split['split'] == 'test']

val_warm = val_df[val_df['userId'].isin(warm_users)].drop(columns=['split'])
test_warm = test_df[test_df['userId'].isin(warm_users)].drop(columns=['split'])

val_warm.to_parquet('/kaggle/working/interim/ratings_val_warm.parquet', index=False)
test_warm.to_parquet('/kaggle/working/interim/ratings_test_warm.parquet', index=False)

print(f"warm users: {len(warm_users_list)}")
print(f"val_warm ratings: {len(val_warm)}")
print(f"test_warm ratings: {len(test_warm)}")

# Update config with everything from this review round
config.update({
    "warm_user_definition": f">= {COLD_START_N} train ratings",
    "val_zero_train_history_pct": 66.2,
    "test_zero_train_history_pct": 75.2,
    "val_warm_user_ratings": int(len(val_warm)),
    "test_warm_user_ratings": int(len(test_warm)),
    "rt_decade_skew_note": "RT coverage concentrated in 2010s (40.4% vs 33.0% catalog share); near-zero pre-1930",
    "content_text_fallback_pct": 32.25,
})
with open('/kaggle/working/interim/week1_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("\nupdated week1_config.json:")
print(json.dumps(config, indent=2))

warm users: 126591
val_warm ratings: 412348
test_warm ratings: 242544

updated week1_config.json:
{
  "train_cutoff": "2017-01-01",
  "val_cutoff": "2018-07-01",
  "cold_start_threshold_N": 25,
  "tmdb_join_coverage_ratings_pct": 98.93,
  "rt_join_coverage_ratings_pct": 66.48,
  "content_text_empty_pct": 4.49,
  "imdb_reviews_final_count": 49582,
  "warm_user_definition": ">= 25 train ratings",
  "val_zero_train_history_pct": 66.2,
  "test_zero_train_history_pct": 75.2,
  "val_warm_user_ratings": 412348,
  "test_warm_user_ratings": 242544,
  "rt_decade_skew_note": "RT coverage concentrated in 2010s (40.4% vs 33.0% catalog share); near-zero pre-1930",
  "content_text_fallback_pct": 32.25
}
